# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [2]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GEMINI_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

API key loaded from Colab Secrets.


In [3]:
!pip install -q google-genai
print("Done.")

Done.


In [4]:
# @title User input needed.

# TODO: Fill in documents in the format mentioned in the first example.

DOCUMENTS = {
    "sample_protocol": (
        "Students must fill in documents in this section. "
        "Documents can be related to any topic. "
        "Each document must contain at least four sentences. "
        "Add a minimum of four documents to test your agent."
    ),

    "tomato_basil_pasta": (
        "Tomato basil pasta is made using pasta, tomatoes, garlic, olive oil and fresh basil. "
        "Boil the pasta in salted water until it becomes al dente. "
        "Prepare the sauce by sauteing garlic and tomatoes in olive oil. "
        "Add fresh basil, mix the cooked pasta with the sauce and serve warm."
    ),

    "vegetable_soup": (
        "Vegetable soup can be prepared using carrots, beans, peas, onions and potatoes. "
        "Chop all vegetables into small pieces before cooking. "
        "Saute the onions and vegetables in a little butter or olive oil. "
        "Add vegetable stock, simmer until the vegetables are tender and season with salt and pepper."
    ),

    "chocolate_cake": (
        "Chocolate cake requires flour, cocoa powder, sugar, eggs, butter and milk. "
        "Mix the dry ingredients in a bowl and combine the wet ingredients separately. "
        "Gradually combine both mixtures until a smooth batter forms. "
        "Bake in a preheated oven until a toothpick inserted into the centre comes out clean."
    ),

    "margherita_pizza": (
        "Margherita pizza is traditionally prepared with pizza dough, tomato sauce, mozzarella and fresh basil. "
        "Roll the dough into a thin circular base. "
        "Spread tomato sauce evenly and add mozzarella cheese. "
        "Bake in a hot oven until the crust becomes golden and the cheese melts, then garnish with basil."
    )
}


In [6]:
# @title Define a tool
def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check for regex match.
print(search_documents("minimum of 4"))


[Source: sample_protocol]
Students must fill in documents in this section. Documents can be related to any topic. Each document must contain at least four sentences. Add a minimum of four documents to test your agent.


In [15]:
# @title Ask Gemini to search for something
from google import genai
import time
import os

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

model = "gemini-3.5-flash-lite"

print("Model ready:", model)

Model ready: gemini-3.5-flash-lite


In [16]:
import time
from google import genai
from google.genai import types
import os

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_documents",
        description="Search the documents to find the most relevant paragraph.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)


def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=model,
                contents=contents,
                config=config
            )
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"  Rate limit hit, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        contents.append(model_content)

        calls = [p.function_call for p in model_content.parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")
            result = search_documents(**args)
            print(f"  Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )
        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"


_ = run_agent("What topics are students supposed to share documents on?")

Question: What topics are students supposed to share documents on?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'topics students share documents on'})
  Result preview: [Source: sample_protocol]
Students must fill in documents in this section. Documents can be related ...

Final answer:
Based on the provided documents, students can share documents related to **any topic**. There are no specific restrictions on the subjects they must choose.


In [17]:
# @title Ask follow up questions

# TODO: Enter questions based on documents for the agent to answer.


questions = [
    # Answer should be found in the documents
    "How long does it take to cook tomato basil pasta?",

    # Another question answered by the documents
    "What ingredients are needed to make chocolate cake?",

    # Not answered by the documents
    "How many ghosts eat carrots everyday?",

    # General knowledge question
    "What is the most popular food item in Canada?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)
    print("\n" + "=" * 60)
    run_agent(q)

assert(len(questions) >= 4)
assert("Add a question here." not in questions)


Question: How long does it take to cook tomato basil pasta?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'tomato basil pasta cook time'})
  Result preview: [Source: tomato_basil_pasta]
Tomato basil pasta is made using pasta, tomatoes, garlic, olive oil and...

Final answer:
Based on standard cooking times for pasta and a quick fresh tomato-basil sauce, it generally takes about **15 to 20 minutes** total:

* **Boiling the pasta:** 8 to 12 minutes (depending on the type of pasta and package instructions) to reach *al dente*.
* **Making the sauce:** 5 to 7 minutes to sauté the garlic and tomatoes in olive oil and toss everything together with fresh basil.

Question: What ingredients are needed to make chocolate cake?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'chocolate cake ingredients'})
  Result preview: [Source: chocolate_cake]
Chocolate cake requires flour, cocoa powder, sugar

In [ ]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)